# 모두몰 요약 리포트 — SQL 분석

**과제**: 모두몰 데이터를 요약하는 분석 질문 5개 + SQL + 인사이트

**환경 안내**: 원본 문제는 BigQuery 문법(`project_name.dataset_name.table`)으로 제시되었지만,
실행 편의를 위해 로컬에서 **DuckDB**로 동일한 스키마/데이터를 구성해 실행했습니다.
문법은 BigQuery Standard SQL과 거의 동일하므로, 실제 BigQuery 환경에 옮길 때는
테이블 참조 경로(`project.dataset.table`)만 맞춰주면 그대로 사용 가능합니다.

**스키마 한계 (미리 짚어둠)**: `orders` 테이블에 `product_id`가 없어 `orders`와 `products`를
JOIN할 수 없습니다. 따라서 "카테고리별 매출" 같은 질문은 만들 수 없고,
`products`는 가격 분포 등 단독 분석으로만 다뤘습니다.


## 0. 환경 설정 및 테이블 생성 (DuckDB)

In [6]:

! pip install duckdb pandas

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ------------- -------------------------- 4.5/13.2 MB 22.3 MB/s eta 0:00:01
   ----------------------------- ---------- 9.7/13.2 MB 24.1 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 22.3 MB/s  0:00:00


In [7]:
import duckdb
import pandas as pd

con = duckdb.connect(database=":memory:")

con.execute("""
CREATE OR REPLACE TABLE customers (
    customer_id STRING,
    name STRING,
    country STRING,
    signup_date DATE,
    grade STRING
);
""")

con.execute("""
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');
""")

con.execute("""
CREATE OR REPLACE TABLE orders (
    order_id STRING,
    customer_id STRING,
    order_date DATE,
    status STRING,
    amount DECIMAL(12,2)
);
""")

con.execute("""
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")

con.execute("""
CREATE OR REPLACE TABLE products (
    product_id STRING,
    product_name STRING,
    category STRING,
    price DECIMAL(12,2)
);
""")

con.execute("""
INSERT INTO products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000);
""")

print("테이블 생성 완료: customers, orders, products")


테이블 생성 완료: customers, orders, products


## Q1. 월별 매출 추이는? (결제 확정 기준: Paid, Shipped)

- **사용 조건**: GROUP BY + 집계함수, ORDER BY


In [11]:
q1 = con.sql("""
SELECT
    EXTRACT(YEAR FROM order_date) AS 년도,
    EXTRACT(MONTH FROM order_date) AS 월,
    COUNT(*) AS 주문수,
    SUM(amount) AS 월별매출
FROM orders
WHERE status IN ('Paid', 'Shipped')
GROUP BY 년도, 월
ORDER BY 년도, 월
""").df()
q1

,년도,월,주문수,월별매출
0,2023,9,4,511000.0
1,2023,10,4,743000.0
2,2023,11,4,725000.0
3,2023,12,4,488000.0
4,2024,1,4,850000.0
5,2024,2,1,187000.0


> **인사이트**: Paid/Shipped 기준 매출은 9월~11월까지 대체로 증가하는 흐름을 보이나, 월별 표본이 3~6건 수준으로 작아 추세로 확정하기는 이르다 (추정치, 검증 필요).

## Q2. 주문 상태별 건수와 매출 비중은?

- **사용 조건**: GROUP BY + 집계함수, CASE WHEN, HAVING, ORDER BY

[문제] orders에서 주문 상태(status)별로 건수와 매출을 요약해 보세요.

상태를 두 그룹으로 분류하세요
Paid, Shipped → 매출인정
Cancelled, Returned → 취소_반품
그 외 (Placed 등) → 진행중
status와 방금 만든 그룹 두 기준으로 각각 건수(COUNT)와 총금액(SUM) 을 구하세요
건수가 2건 이상인 그룹만 결과에 남기세요


In [16]:
q2 = con.sql("""
SELECT
    status,
    CASE
        WHEN status IN ('Paid', 'Shipped') THEN '매출인정'
        WHEN status IN ('Cancelled', 'Returned') THEN '취소_반품'
        ELSE '진행중'
    END AS status_group,
    COUNT(*) AS order_cnt,
    SUM(amount) AS total_amount
FROM orders
GROUP BY status, status_group
HAVING COUNT(*) >= 2
ORDER BY total_amount DESC
""").df()
q2


,status,status_group,order_cnt,total_amount
0,Paid,매출인정,13,1909000.0
1,Shipped,매출인정,8,1595000.0
2,Returned,취소_반품,3,179000.0
3,Placed,진행중,3,123000.0
4,Cancelled,취소_반품,3,NaN


> **인사이트**: Cancelled 3건, Returned 3건으로 취소/반품 비중이 낮지 않아 원인 파악이 필요해 보이나, 이 결과만으로 원인을 단정할 수는 없다 (상관 수준, 인과관계 아님).

Q3. [orders] 고객별 총 주문금액을 요약해보세요

금액(amount)이 있는 주문만 대상으로
고객별(customer_id) 총 주문금액을 구하고
총액이 20만원 이상인 고객만 남기고 (HAVING)
총액이 큰 순으로 정렬하세요


In [17]:
q3 = con.sql("""
SELECT
    customer_id,
    COUNT(*) AS 주문수,
    SUM(amount) AS 총주문금액
FROM orders
WHERE amount IS NOT NULL
GROUP BY customer_id
HAVING SUM(amount) >= 200000
ORDER BY 총주문금액 DESC
""").df()
q3


,customer_id,주문수,총주문금액
0,C013,2,725000.0
1,C007,2,678000.0
2,C001,4,380000.0
3,C012,2,362000.0
4,C003,2,349000.0
5,C008,2,314000.0
6,C002,3,308000.0


> **인사이트**: 주문 금액이 많다고 주문수가 많은건 아님 

Q4. [customers] 고객 등급(grade)별 고객수를 요약해보세요

등급별로 고객 수(COUNT)를 구하고
고객이 2명 이상인 등급만 남기고 (HAVING)
고객 수가 많은 순으로 정렬하세요

In [18]:
q4 = con.sql("""

SELECT
    grade,
    COUNT(*) AS 고객수
FROM customers
GROUP BY grade
HAVING COUNT(*) >= 2
ORDER BY 고객수 DESC
""").df()
q4


,grade,고객수
0,Bronze,5
1,Silver,5
2,Gold,4


> **인사이트**: 브론즈 등급 고객이 제일 많음

Q5. [products] 카테고리별 상품 요약을 만들어보세요

products를 카테고리별로 묶어
카테고리명, 상품 수(COUNT), 평균 가격(AVG), 최고 가격(MAX)을 구하고
상품이 2개 이상인 카테고리만 남기고 (HAVING)
평균 가격이 높은 순으로 정렬하세요


In [20]:
q5 = con.sql("""

SELECT
    category,
    COUNT(*) AS 상품수,
    ROUND(AVG(price), 0) AS 평균가격,
    MAX(price) AS 최고가격
FROM products
GROUP BY category
HAVING COUNT(*) >= 2
ORDER BY 평균가격 DESC
""").df()
q5


,category,상품수,평균가격,최고가격
0,Boots,2,160000.0,175000.0
1,Running,2,104000.0,119000.0
2,Sneakers,2,57000.0,65000.0
3,Sandals,2,31500.0,38000.0
4,Accessory,2,8500.0,12000.0


> **인사이트**: 평균가는 Boots > Running > Sneakers > Sandals > Accessory 순으로 높다. 이는 가격표 상의 분포일 뿐, 어떤 카테고리가 실제로 잘 팔리는지와는 무관하다 (orders–products 간 연결 키 부재로 판매량 분석 불가).